# Display-screen OCR test

This notebook tests the new OCR path for the digital scale display:

1. Use a YOLOv11 display-screen detector to crop the scale screen.
2. Run OpenCV seven-segment OCR on the cropped display.
3. Run PaddleOCR on the same cropped display.
4. Compare both outputs in a small table and save QC crops.

The blue-HSV display finder is kept only as a fallback when YOLO weights are unavailable or no display box is detected.

In [ ]:
from pathlib import Path
import os
import re
import sys

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yaml
except ImportError:
    yaml = None

plt.rcParams['figure.dpi'] = 120

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
CONFIG_PATH = PROJECT_DIR / 'config.yaml'
sys.path.insert(0, str(PROJECT_DIR))

def load_yaml(path):
    if yaml is None or not Path(path).exists():
        return {}
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}

config = load_yaml(CONFIG_PATH)
print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIG_PATH =', CONFIG_PATH, 'exists=', CONFIG_PATH.exists())

## Parameters

Change `YOLO_DISPLAY_WEIGHTS` after training `YOLOv11_display_train.ipynb`. If the weight is missing, this notebook automatically falls back to the old blue-HSV ROI finder.

In [ ]:
# Input images. Set IMAGE_DIR_OVERRIDE when testing a custom folder.
IMAGE_DIR_OVERRIDE = None
IMAGE_EXTENSION = config.get('input', {}).get('image_extension', '.jpg')
IMAGE_DIR = Path(IMAGE_DIR_OVERRIDE) if IMAGE_DIR_OVERRIDE else Path(config.get('input', {}).get('image_dir', ''))

# Default weight path produced by YOLOv11_display_train.ipynb.
YOLO_DISPLAY_WEIGHTS = PROJECT_DIR / 'runs' / 'display_detect' / 'yolo11_display' / 'weights' / 'best.pt'
YOLO_CONF = 0.25
YOLO_IMGSZ = 1280
YOLO_MAX_DET = 5
YOLO_DEVICE = config.get('ocr', {}).get('device') or config.get('runtime', {}).get('gpu_device', 'cuda:0')

# Display crop padding after YOLO detection.
DISPLAY_EXPAND_X = 0.18
DISPLAY_EXPAND_Y = 0.35

# HSV fallback parameters.
USE_BLUE_HSV_FALLBACK = True
BLUE_HSV_LOWER = (85, 35, 20)
BLUE_HSV_UPPER = (145, 255, 255)
BOTTOM_SEARCH_TOP_RATIO = 0.45
MIN_DISPLAY_AREA_RATIO = 0.0003
MAX_DISPLAY_AREA_RATIO = 0.12
DISPLAY_ASPECT_RANGE = (1.2, 8.5)

# Seven-segment OCR parameters.
SEVEN_SEGMENT_SCALE = 4.0
DIGIT_POLARITY = 'auto'  # auto, dark_on_light, light_on_dark
SEGMENT_ON_THRESHOLD = 0.24
MAX_LOOKUP_DISTANCE = 2
COLUMN_ACTIVE_MIN_FRACTION = 0.018
COLUMN_GAP_MERGE_PX = 8
MIN_DIGIT_HEIGHT_RATIO = 0.30
MIN_DIGIT_WIDTH_RATIO = 0.025
DECIMAL_MAX_HEIGHT_RATIO = 0.25
DECIMAL_MAX_WIDTH_RATIO = 0.09

# PaddleOCR comparison.
PADDLEOCR_ENABLED = True
PADDLEOCR_LANG = 'en'
PADDLEOCR_DEVICE = YOLO_DEVICE

IMAGE_INDEX = 0
BATCH_LIMIT = 30
SAVE_QC = True
QC_DIR = PROJECT_DIR / 'notebooks' / 'ocr_yolo_display_qc'

print('IMAGE_DIR =', IMAGE_DIR)
print('YOLO_DISPLAY_WEIGHTS =', YOLO_DISPLAY_WEIGHTS, 'exists=', YOLO_DISPLAY_WEIGHTS.exists())
print('YOLO_DEVICE =', YOLO_DEVICE)

In [ ]:
def bgr_to_rgb(image_bgr):
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

def show_bgr(image_bgr, title=None, figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(bgr_to_rgb(image_bgr))
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

def show_gray(image, title=None, figsize=(8, 4), cmap='gray'):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

def clamp_box(box, image_shape):
    h, w = image_shape[:2]
    x1, y1, x2, y2 = [int(round(float(v))) for v in box]
    x1 = max(0, min(w - 1, x1))
    y1 = max(0, min(h - 1, y1))
    x2 = max(x1 + 1, min(w, x2))
    y2 = max(y1 + 1, min(h, y2))
    return [x1, y1, x2, y2]

def pad_box(box, image_shape, expand_x=0.18, expand_y=0.35):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1
    return clamp_box([
        x1 - bw * expand_x,
        y1 - bh * expand_y,
        x2 + bw * expand_x,
        y2 + bh * expand_y,
    ], image_shape)

def crop_box(image_bgr, box):
    x1, y1, x2, y2 = box
    return image_bgr[y1:y2, x1:x2].copy()

def find_contours(mask):
    result = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return result[0] if len(result) == 2 else result[1]

def draw_box(image_bgr, box, color=(0, 255, 255), label=None, thickness=3):
    out = image_bgr.copy()
    x1, y1, x2, y2 = box
    cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
    if label:
        cv2.putText(out, str(label), (x1, max(22, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)
    return out

## Display detection: YOLO first, HSV fallback second

In [ ]:
def load_display_yolo(weights_path):
    weights_path = Path(weights_path)
    if not weights_path.exists():
        print('Display YOLO weights not found; HSV fallback will be used:', weights_path)
        return None
    try:
        from ultralytics import YOLO
    except ImportError:
        print('ultralytics is not installed; HSV fallback will be used.')
        return None
    print('Loading display YOLO:', weights_path)
    return YOLO(str(weights_path))

def detect_display_with_yolo(model, image_bgr):
    if model is None:
        return None
    results = model.predict(
        source=image_bgr,
        conf=YOLO_CONF,
        imgsz=YOLO_IMGSZ,
        max_det=YOLO_MAX_DET,
        device=YOLO_DEVICE,
        verbose=False,
    )
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return None

    h, w = image_bgr.shape[:2]
    boxes = results[0].boxes.xyxy.detach().cpu().numpy()
    confs = results[0].boxes.conf.detach().cpu().numpy()
    candidates = []
    for box, conf in zip(boxes, confs):
        raw_box = clamp_box(box, image_bgr.shape)
        x1, y1, x2, y2 = raw_box
        bw = x2 - x1
        bh = y2 - y1
        lower_bonus = (y1 + y2) / max(1.0, 2.0 * h)
        area = bw * bh
        score = float(conf) * (0.8 + lower_bonus) * max(1.0, area ** 0.25)
        candidates.append({
            'source': 'yolo11_display',
            'screen_box': raw_box,
            'padded_box': pad_box(raw_box, image_bgr.shape, DISPLAY_EXPAND_X, DISPLAY_EXPAND_Y),
            'confidence': float(conf),
            'score': float(score),
        })
    return sorted(candidates, key=lambda item: item['score'], reverse=True)[0]

def find_blue_display_roi(image_bgr):
    h, w = image_bgr.shape[:2]
    y0 = int(round(h * BOTTOM_SEARCH_TOP_RATIO))
    search = image_bgr[y0:h]
    hsv = cv2.cvtColor(search, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(BLUE_HSV_LOWER, dtype=np.uint8), np.array(BLUE_HSV_UPPER, dtype=np.uint8))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    candidates = []
    image_area = float(h * w)
    lo_aspect, hi_aspect = DISPLAY_ASPECT_RANGE
    for contour in find_contours(mask):
        x, y, bw, bh = cv2.boundingRect(contour)
        if bw <= 0 or bh <= 0:
            continue
        box_area = float(bw * bh)
        area_ratio = box_area / image_area
        aspect = float(bw) / float(bh)
        if area_ratio < MIN_DISPLAY_AREA_RATIO or area_ratio > MAX_DISPLAY_AREA_RATIO:
            continue
        if aspect < lo_aspect or aspect > hi_aspect:
            continue
        screen_box = [x, y0 + y, x + bw, y0 + y + bh]
        fill = float(cv2.contourArea(contour)) / max(1.0, box_area)
        lower_bonus = (y0 + y + 0.5 * bh) / max(1.0, h)
        score = box_area * (0.5 + fill) * (0.6 + lower_bonus)
        candidates.append({
            'source': 'blue_hsv_fallback',
            'screen_box': screen_box,
            'padded_box': pad_box(screen_box, image_bgr.shape, DISPLAY_EXPAND_X, DISPLAY_EXPAND_Y),
            'confidence': None,
            'score': float(score),
        })
    if candidates:
        return sorted(candidates, key=lambda item: item['score'], reverse=True)[0]
    return None

def detect_display_roi(image_bgr, yolo_model=None):
    det = detect_display_with_yolo(yolo_model, image_bgr)
    if det is not None:
        return det
    if USE_BLUE_HSV_FALLBACK:
        return find_blue_display_roi(image_bgr)
    return None

## Seven-segment OCR

In [ ]:
DIGITS_LOOKUP = {
    (1, 1, 1, 0, 1, 1, 1): '0',
    (0, 0, 1, 0, 0, 1, 0): '1',
    (1, 0, 1, 1, 1, 0, 1): '2',
    (1, 0, 1, 1, 0, 1, 1): '3',
    (0, 1, 1, 1, 0, 1, 0): '4',
    (1, 1, 0, 1, 0, 1, 1): '5',
    (1, 1, 0, 1, 1, 1, 1): '6',
    (1, 0, 1, 0, 0, 1, 0): '7',
    (1, 1, 1, 1, 1, 1, 1): '8',
    (1, 1, 1, 1, 0, 1, 1): '9',
}

def normalize_numeric_text(text):
    text = str(text or '').strip()
    text = text.translate(str.maketrans({'O': '0', 'o': '0', 'I': '1', 'l': '1', '|': '1', 'B': '8', 'S': '5', ',': '.'}))
    text = re.sub(r'[^0-9.]', '', text)
    if text.count('.') > 1:
        first = text.find('.')
        text = text[:first + 1] + text[first + 1:].replace('.', '')
    return text

def prepare_digit_mask(display_crop, polarity):
    scaled = cv2.resize(display_crop, None, fx=SEVEN_SEGMENT_SCALE, fy=SEVEN_SEGMENT_SCALE, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(scaled, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
    if polarity == 'dark_on_light':
        _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return mask, gray

def active_column_groups(mask):
    active_counts = np.sum(mask > 0, axis=0)
    threshold = max(2, int(round(mask.shape[0] * COLUMN_ACTIVE_MIN_FRACTION)))
    active = np.where(active_counts >= threshold)[0]
    if len(active) == 0:
        return []
    groups = []
    start = int(active[0])
    prev = int(active[0])
    for x in active[1:]:
        x = int(x)
        if x - prev <= COLUMN_GAP_MERGE_PX:
            prev = x
        else:
            groups.append([start, prev + 1])
            start = x
            prev = x
    groups.append([start, prev + 1])
    return groups

def extract_digit_tokens(mask):
    h, w = mask.shape[:2]
    tokens = []
    for x1, x2 in active_column_groups(mask):
        x1 = max(0, x1 - 2)
        x2 = min(w, x2 + 2)
        sub = mask[:, x1:x2]
        ys, xs = np.where(sub > 0)
        if len(xs) == 0:
            continue
        y1 = int(max(0, ys.min() - 2))
        y2 = int(min(h, ys.max() + 3))
        bw = x2 - x1
        bh = y2 - y1
        is_decimal = (
            bh <= h * DECIMAL_MAX_HEIGHT_RATIO
            and bw <= w * DECIMAL_MAX_WIDTH_RATIO
            and (y1 + y2) * 0.5 >= h * 0.45
        )
        if is_decimal:
            tokens.append({'type': 'decimal', 'box': [x1, y1, x2, y2], 'text': '.'})
            continue
        if bh < h * MIN_DIGIT_HEIGHT_RATIO or bw < w * MIN_DIGIT_WIDTH_RATIO:
            continue
        tokens.append({'type': 'digit', 'box': [x1, y1, x2, y2]})
    return sorted(tokens, key=lambda item: item['box'][0])

def recognize_digit_from_mask(mask, box):
    x1, y1, x2, y2 = box
    h_full, w_full = mask.shape[:2]
    pad_x = max(2, int(round((x2 - x1) * 0.08)))
    pad_y = max(2, int(round((y2 - y1) * 0.06)))
    x1 = max(0, x1 - pad_x)
    x2 = min(w_full, x2 + pad_x)
    y1 = max(0, y1 - pad_y)
    y2 = min(h_full, y2 + pad_y)
    roi = mask[y1:y2, x1:x2]
    h, w = roi.shape[:2]
    if h < 8 or w < 4:
        return '?', None, None

    dW = max(1, int(w * 0.26))
    dH = max(1, int(h * 0.18))
    dHC = max(1, int(h * 0.12))
    segments = [
        ((0, 0), (w, dH)),
        ((0, 0), (dW, h // 2)),
        ((w - dW, 0), (w, h // 2)),
        ((0, (h - dHC) // 2), (w, (h + dHC) // 2)),
        ((0, h // 2), (dW, h)),
        ((w - dW, h // 2), (w, h)),
        ((0, h - dH), (w, h)),
    ]
    status = []
    for (sx1, sy1), (sx2, sy2) in segments:
        seg = roi[max(0, sy1):min(h, sy2), max(0, sx1):min(w, sx2)]
        total = max(1, seg.shape[0] * seg.shape[1])
        on_fraction = cv2.countNonZero(seg) / float(total)
        status.append(1 if on_fraction > SEGMENT_ON_THRESHOLD else 0)
    status_tuple = tuple(status)
    if status_tuple in DIGITS_LOOKUP:
        return DIGITS_LOOKUP[status_tuple], status_tuple, 0
    distances = [(sum(a != b for a, b in zip(status_tuple, key)), value) for key, value in DIGITS_LOOKUP.items()]
    distance, digit = sorted(distances, key=lambda item: item[0])[0]
    if distance <= MAX_LOOKUP_DISTANCE:
        return digit, status_tuple, int(distance)
    return '?', status_tuple, int(distance)

def recognize_seven_segment(display_crop):
    polarities = ['dark_on_light', 'light_on_dark'] if DIGIT_POLARITY == 'auto' else [DIGIT_POLARITY]
    attempts = []
    for polarity in polarities:
        mask, gray = prepare_digit_mask(display_crop, polarity)
        tokens = extract_digit_tokens(mask)
        chars = []
        digit_count = 0
        unknown_count = 0
        distance_sum = 0
        for token in tokens:
            if token['type'] == 'decimal':
                chars.append('.')
                continue
            digit, status, distance = recognize_digit_from_mask(mask, token['box'])
            token['text'] = digit
            token['segments'] = status
            token['distance'] = distance
            chars.append(digit)
            if digit == '?':
                unknown_count += 1
            else:
                digit_count += 1
                distance_sum += int(distance or 0)
        raw_text = ''.join(chars)
        numeric_text = normalize_numeric_text(raw_text)
        score = digit_count * 10 - unknown_count * 6 - distance_sum + (4 if numeric_text else 0)
        attempts.append({
            'method': 'seven_segment',
            'polarity': polarity,
            'text': numeric_text,
            'raw_text': raw_text,
            'score': float(score),
            'mask': mask,
            'gray': gray,
            'tokens': tokens,
            'digit_count': digit_count,
            'unknown_count': unknown_count,
        })
    attempts = sorted(attempts, key=lambda item: item['score'], reverse=True)
    return attempts[0] if attempts else {'method': 'seven_segment', 'text': '', 'raw_text': '', 'score': 0, 'tokens': []}

## PaddleOCR comparison

In [ ]:
paddle_ocr = None

def load_paddleocr_reader():
    global paddle_ocr
    if not PADDLEOCR_ENABLED:
        return None
    if paddle_ocr is not None:
        return paddle_ocr
    try:
        from paddleocr import PaddleOCR
    except ImportError:
        print('paddleocr is not installed; PaddleOCR comparison will be skipped.')
        return None
    try:
        from utils.device import paddleocr_device_kwargs
        device_kwargs = paddleocr_device_kwargs(PADDLEOCR_DEVICE)
    except Exception:
        device_kwargs = {'use_gpu': str(PADDLEOCR_DEVICE).lower() != 'cpu'}
    print('Loading PaddleOCR on', PADDLEOCR_DEVICE, device_kwargs)
    try:
        paddle_ocr = PaddleOCR(use_angle_cls=False, lang=PADDLEOCR_LANG, show_log=False, **device_kwargs)
    except TypeError:
        device_kwargs.pop('gpu_id', None)
        paddle_ocr = PaddleOCR(use_angle_cls=False, lang=PADDLEOCR_LANG, show_log=False, **device_kwargs)
    return paddle_ocr

def flatten_paddle_result(result):
    if not result:
        return []
    lines = result[0] if isinstance(result, list) and result and isinstance(result[0], list) else result
    out = []
    for line in lines:
        try:
            text = str(line[1][0]).strip()
            conf = float(line[1][1])
            out.append({'text': text, 'confidence': conf})
        except Exception:
            continue
    return out

def read_with_paddleocr(display_crop, reader=None):
    reader = reader or load_paddleocr_reader()
    if reader is None:
        return {'method': 'paddleocr', 'text': '', 'raw_text': '', 'confidence': None, 'lines': []}
    variants = []
    variants.append(('crop', display_crop))
    up = cv2.resize(display_crop, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_CUBIC)
    variants.append(('upscaled', up))
    gray = cv2.cvtColor(up, cv2.COLOR_BGR2GRAY)
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
    _, inv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    variants.append(('threshold_inv', cv2.cvtColor(inv, cv2.COLOR_GRAY2BGR)))

    candidates = []
    for variant_name, image in variants:
        try:
            result = reader.ocr(image, cls=False)
        except Exception as exc:
            candidates.append({'variant': variant_name, 'text': '', 'raw_text': f'ERROR: {exc}', 'confidence': None, 'lines': []})
            continue
        lines = flatten_paddle_result(result)
        raw_text = ' '.join(item['text'] for item in lines)
        text = normalize_numeric_text(raw_text)
        confs = [item['confidence'] for item in lines]
        confidence = float(np.mean(confs)) if confs else None
        score = len(text) * 8 + (confidence or 0.0) * 5
        candidates.append({'variant': variant_name, 'text': text, 'raw_text': raw_text, 'confidence': confidence, 'lines': lines, 'score': score})
    candidates = sorted(candidates, key=lambda item: item.get('score', 0), reverse=True)
    best = candidates[0] if candidates else {'text': '', 'raw_text': '', 'confidence': None, 'lines': []}
    best['method'] = 'paddleocr'
    best['attempts'] = candidates
    return best

## Load images and models

In [ ]:
image_paths = sorted(Path(IMAGE_DIR).glob(f'*{IMAGE_EXTENSION}')) if IMAGE_DIR.exists() else []
print('image count =', len(image_paths))
if image_paths:
    print('first image =', image_paths[0])

display_yolo = load_display_yolo(YOLO_DISPLAY_WEIGHTS)
paddle_reader = load_paddleocr_reader() if PADDLEOCR_ENABLED else None

if SAVE_QC:
    QC_DIR.mkdir(parents=True, exist_ok=True)
    print('QC_DIR =', QC_DIR)

## One-image visual test

In [ ]:
def run_one_image(image_path, show=True, save_qc=SAVE_QC):
    image_path = Path(image_path)
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return {'image_name': image_path.name, 'error': 'could_not_read_image'}

    det = detect_display_roi(image_bgr, display_yolo)
    if det is None:
        return {'image_name': image_path.name, 'error': 'display_not_found'}

    display_crop = crop_box(image_bgr, det['padded_box'])
    seven = recognize_seven_segment(display_crop)
    paddle = read_with_paddleocr(display_crop, paddle_reader) if PADDLEOCR_ENABLED else {'text': '', 'raw_text': '', 'confidence': None}

    if save_qc:
        stem = image_path.stem
        crop_path = QC_DIR / f'{stem}_display_crop.jpg'
        mask_path = QC_DIR / f'{stem}_seven_segment_mask.png'
        overlay_path = QC_DIR / f'{stem}_display_box.jpg'
        cv2.imwrite(str(crop_path), display_crop)
        if seven.get('mask') is not None:
            cv2.imwrite(str(mask_path), seven['mask'])
        overlay = draw_box(image_bgr, det['screen_box'], color=(0, 0, 255), label='raw display')
        overlay = draw_box(overlay, det['padded_box'], color=(0, 255, 255), label=det['source'])
        cv2.imwrite(str(overlay_path), overlay)

    if show:
        overlay = draw_box(image_bgr, det['screen_box'], color=(0, 0, 255), label='raw display')
        overlay = draw_box(overlay, det['padded_box'], color=(0, 255, 255), label=det['source'])
        show_bgr(overlay, f'{image_path.name} | {det["source"]}', figsize=(9, 7))
        show_bgr(display_crop, 'display crop', figsize=(8, 3))
        if seven.get('mask') is not None:
            show_gray(seven['mask'], f'seven-segment mask | polarity={seven.get("polarity")}', figsize=(8, 3))
        print('seven_segment:', seven.get('text'), '| raw:', seven.get('raw_text'), '| score:', seven.get('score'))
        print('paddleocr    :', paddle.get('text'), '| raw:', paddle.get('raw_text'), '| conf:', paddle.get('confidence'))

    return {
        'image_name': image_path.name,
        'display_source': det.get('source'),
        'display_confidence': det.get('confidence'),
        'display_box': det.get('screen_box'),
        'display_padded_box': det.get('padded_box'),
        'seven_segment_text': seven.get('text', ''),
        'seven_segment_raw': seven.get('raw_text', ''),
        'seven_segment_polarity': seven.get('polarity', ''),
        'seven_segment_score': seven.get('score'),
        'paddleocr_text': paddle.get('text', ''),
        'paddleocr_raw': paddle.get('raw_text', ''),
        'paddleocr_confidence': paddle.get('confidence'),
        'error': '',
    }

assert image_paths, 'No images found. Check IMAGE_DIR and IMAGE_EXTENSION.'
single_result = run_one_image(image_paths[IMAGE_INDEX], show=True)
pd.DataFrame([single_result])

## Batch comparison

In [ ]:
def run_batch(image_paths, limit=BATCH_LIMIT):
    rows = []
    for idx, image_path in enumerate(image_paths[:limit], start=1):
        print(f'[{idx}/{min(limit, len(image_paths))}] {Path(image_path).name}')
        try:
            row = run_one_image(image_path, show=False, save_qc=SAVE_QC)
        except Exception as exc:
            row = {'image_name': Path(image_path).name, 'error': str(exc)}
        rows.append(row)
    df = pd.DataFrame(rows)
    if SAVE_QC:
        csv_path = QC_DIR / 'ocr_display_comparison.csv'
        df.to_csv(csv_path, index=False)
        print('saved ->', csv_path)
    return df

batch_df = run_batch(image_paths, limit=BATCH_LIMIT)
batch_df

## Inspect disagreement cases

Use this cell after the batch run to quickly inspect images where seven-segment OCR and PaddleOCR disagree.

In [ ]:
if 'batch_df' in globals() and not batch_df.empty:
    compare = batch_df.fillna('')
    disagreement = compare[compare['seven_segment_text'].astype(str) != compare['paddleocr_text'].astype(str)]
    print('disagreement count =', len(disagreement))
    display(disagreement[['image_name', 'display_source', 'seven_segment_text', 'seven_segment_raw', 'paddleocr_text', 'paddleocr_raw', 'error']].head(30))
else:
    print('Run the batch cell first.')